# ALG — isolated 168,960-configuration + neural-test reproduction

This notebook is a protected copy of the user-edited figure notebook. It keeps the same figure functions and visual choices, but reads the final 168,960-configuration campaign, appends the three neural-model representation datasets to the held-out test where compatible, and reads the real controlled ALG* results. All outputs are written below `generated_168960_neural_test/`; the original notebook and figures are never modified.


## 1. Exact analysis and plotting source

In [ ]:
#!/usr/bin/env python3
"""Generate every figure used by the ALG Information Systems manuscript.

The figures deliberately have no internal title: the LaTeX captions carry the
explanation. All baseline methods are retained in the exhaustive comparisons.
"""

from __future__ import annotations

from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, TwoSlopeNorm, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon


_start = Path.cwd().resolve()
PROJECT = next(
    (candidate for candidate in (_start, *_start.parents)
     if (candidate / "outputs").is_dir()
     and (candidate / "reports" / "alg_information_systems_current_20260925").is_dir()),
    None,
)
if PROJECT is None:
    raise RuntimeError("Could not locate the metamatch project from the current notebook directory.")
SOURCE_ROOT = PROJECT / "reports" / "alg_information_systems_current_20260925"
ROOT = SOURCE_ROOT / "generated_168960_neural_test"
METADATA = SOURCE_ROOT / "data_revision"
DATA = ROOT / "data_revision"
TABLES = ROOT / "tables"
FIGURES = ROOT / "figures"
AGGREGATE = PROJECT / "outputs" / "alg_framework_results_168960_243_aggregate"
NEURAL = PROJECT / "outputs" / "alg_neural_external_results_168960_aggregate"
SYNTHETIC = PROJECT / "outputs" / "naeem2020_toy_all25_grid1320_128seeds_fixed_aggregate" / "naeem2020_all27_mean_se.csv"
CONTROLLED_RAW = PROJECT / "outputs" / "alg_literature_synthetic_algstar_128seeds_aggregate" / "all_scores.csv"
for directory in (ROOT, DATA, TABLES, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

ALG_STAR = "algfw_L-local_scaling_manhattan_k3_G-centroid_cosine_exponential_scale2_lambda0.7"
INITIAL_ALG = "overlap_cosine_plus_pooled_cosine"
PURPLE = "#6D28D9"
ORANGE = "#E67E22"
BLUE = "#3B78A8"
GRAY = "#9AA0A6"
GRID = "#DCE1E6"

DOMAIN_COLORS = {
    "ALG": PURPLE,
    "General-purpose": "#8C959F",
    "Generative-model evaluation": "#E67E22",
    "Optimal transport": "#2F6DAE",
    "Point-set / 3D": "#2E8B57",
    "Image matching": "#C44E9B",
    "Spectral clustering": "#1B9E9A",
    "Topology": "#8C6D31",
}

GENERATIVE = {
    "prdc_density_sym", "kid_polynomial_similarity", "coverage_sym_k5",
    "prd_f1_max", "prdc_f1", "manifold_pr_f1", "prc_f1", "prdc_coverage",
    "clipped_coverage", "beta_recall_similarity", "gaussian_frechet_similarity",
    "alpha_precision_similarity", "prdc_density", "prc_precision_cover",
    "clipped_density", "prc_recall_cover", "prdc_recall", "improved_recall",
    "prdc_precision", "improved_precision", "geometry_score_similarity",
    "authenticity_similarity",
}
OPTIMAL_TRANSPORT = {
    "sliced_wasserstein_similarity", "emd_wasserstein_similarity",
    "sinkhorn_divergence_similarity", "gromov_wasserstein_similarity",
}
POINT_SET = {
    "modified_hausdorff_similarity", "chamfer_similarity",
    "hausdorff95_similarity", "dcd_similarity", "hausdorff_similarity",
    "point_fscore_global",
}


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    """Holm family-wise correction, implemented explicitly for reproducibility."""
    values = np.asarray(p_values, dtype=float)
    order = np.argsort(values)
    adjusted = np.empty_like(values)
    running = 0.0
    total = len(values)
    for rank, index in enumerate(order):
        running = max(running, min(1.0, (total - rank) * values[index]))
        adjusted[index] = running
    return adjusted


def bootstrap_mean_ci(
    values: np.ndarray,
    rng: np.random.Generator,
    repetitions: int = 10_000,
) -> tuple[float, float]:
    """Percentile bootstrap confidence interval for a paired mean difference."""
    values = np.asarray(values, dtype=float)
    indices = rng.integers(0, len(values), size=(repetitions, len(values)))
    means = values[indices].mean(axis=1)
    low, high = np.quantile(means, [0.025, 0.975])
    return float(low), float(high)


def rebuild_article_results(
    bootstrap_repetitions: int = 10_000,
    seed: int = 20260925,
) -> dict[str, pd.DataFrame]:
    """Rebuild the article tables on 183 benchmark + 3 neural test datasets."""
    roles = pd.read_csv(METADATA / "method_roles_40baselines.csv")
    included = roles.loc[roles["include_in_baseline_comparison"].astype(bool)].copy()

    selected = pd.read_csv(AGGREGATE / "frozen_global_selection_test183_all_metrics.csv")
    baselines_all = pd.read_csv(AGGREGATE / "baseline_results_test183.csv.gz")
    selected = selected[["support", "dataset", "metric", "f1"]]
    comparison_metrics = set(included["metric"]) - {ALG_STAR}
    baselines = baselines_all.loc[
        baselines_all["metric"].isin(comparison_metrics),
        ["support", "dataset", "metric", "f1"],
    ]

    neural = pd.read_csv(NEURAL / "all_methods_neural_per_dataset.csv")
    neural = neural.loc[
        neural["metric"].isin(set(included["metric"])),
        ["support", "dataset", "metric", "f1"],
    ].copy()
    neural["support"] = "neural_models"

    primary = pd.concat([selected, baselines, neural], ignore_index=True)
    primary = primary.merge(
        included[["metric", "display_name", "role", "provenance"]],
        on="metric", how="left", validate="many_to_one",
    )
    primary.to_csv(DATA / "primary_method_results_test186_40baselines.csv", index=False)

    expected_by_support = primary.loc[primary["metric"].eq(ALG_STAR)].groupby("support")["dataset"].nunique().to_dict()
    support_order = ["graphs", "images", "openml", "tabular", "text", "time_series", "neural_models"]
    summary_records: list[dict[str, object]] = []
    for metric, part in primary.groupby("metric", sort=False):
        meta = included.loc[included["metric"].eq(metric)].iloc[0]
        record: dict[str, object] = {
            "metric": metric,
            "display_name": meta["display_name"],
            "role": meta["role"],
            "overall_mean_f1": part["f1"].mean(),
            "overall_n": part["dataset"].nunique(),
        }
        for support in support_order:
            support_part = part.loc[part["support"].eq(support)]
            record[f"{support}_mean_f1"] = support_part["f1"].mean()
            record[f"{support}_n"] = support_part["dataset"].nunique()
            record[f"{support}_expected_n"] = expected_by_support.get(support, 0)
        summary_records.append(record)
    support_summary = (pd.DataFrame(summary_records)
                       .sort_values("overall_mean_f1", ascending=False)
                       .reset_index(drop=True))
    support_summary.to_csv(DATA / "all_primary_methods_by_support_test186_40baselines.csv", index=False)

    family_catalog = pd.read_csv(METADATA / "baseline_catalog_47.csv")[["metric", "family"]]
    comparator = support_summary.merge(family_catalog, on="metric", how="left")
    comparator.loc[comparator["metric"].isin([ALG_STAR, INITIAL_ALG]), "family"] = "ALG"
    comparator.insert(len(comparator.columns), "rank", np.arange(1, len(comparator) + 1))
    comparator.to_csv(DATA / "complete_comparator_suite_test186_42.csv", index=False)

    available_sets = [set(part["dataset"]) for _, part in primary.groupby("metric")]
    common_datasets = set.intersection(*available_sets)
    common = (primary.loc[primary["dataset"].isin(common_datasets)]
              .groupby(["metric", "display_name", "role"], as_index=False)
              .agg(common_n=("dataset", "nunique"), mean_f1_common=("f1", "mean"))
              .sort_values("mean_f1_common", ascending=False))
    common.to_csv(DATA / "common_all_42_methods.csv", index=False)

    # Recompute macro means and fractional wins on the complete 186-dataset test.
    best = primary.groupby("dataset")["f1"].transform("max")
    primary_wins = primary.assign(is_best=np.isclose(primary["f1"], best, atol=1e-12))
    primary_wins["tie_size"] = primary_wins.groupby("dataset")["is_best"].transform("sum")
    primary_wins["fractional_win"] = np.where(
        primary_wins["is_best"], 1.0 / primary_wins["tie_size"], 0.0
    )
    leaderboard = (primary_wins.groupby("metric", as_index=False)
                   .agg(n_datasets=("dataset", "nunique"),
                        mean_f1=("f1", "mean"), median_f1=("f1", "median"),
                        n_dataset_wins=("fractional_win", "sum"),
                        n_strict_wins=("tie_size", lambda x: int(((x == 1) & primary_wins.loc[x.index, "is_best"]).sum())),
                        n_tied_wins=("is_best", lambda x: int((x & (primary_wins.loc[x.index, "tie_size"] > 1)).sum())))
                   .sort_values("mean_f1", ascending=False))
    leaderboard.to_csv(DATA / "confirmatory_selected_alg_plus_baselines_test186.csv", index=False)

    rng = np.random.default_rng(seed)
    baseline_metrics = included.loc[
        ~included["metric"].isin([ALG_STAR, INITIAL_ALG]), "metric"
    ].tolist()
    paired_records: list[dict[str, object]] = []
    for alg_metric in [ALG_STAR, INITIAL_ALG]:
        alg_meta = included.loc[included["metric"].eq(alg_metric)].iloc[0]
        alg = primary.loc[primary["metric"].eq(alg_metric)].drop_duplicates("dataset").set_index("dataset")["f1"]
        raw_p_values: list[float] = []
        block: list[dict[str, object]] = []
        for baseline_metric in baseline_metrics:
            baseline_meta = included.loc[included["metric"].eq(baseline_metric)].iloc[0]
            baseline = primary.loc[primary["metric"].eq(baseline_metric)].drop_duplicates("dataset").set_index("dataset")["f1"]
            common_index = alg.index.intersection(baseline.index)
            delta = alg.loc[common_index].to_numpy() - baseline.loc[common_index].to_numpy()
            p_value = 1.0 if np.allclose(delta, 0) else float(
                wilcoxon(delta, zero_method="wilcox", alternative="two-sided").pvalue
            )
            ci_low, ci_high = bootstrap_mean_ci(delta, rng, bootstrap_repetitions)
            raw_p_values.append(p_value)
            block.append({
                "alg_metric": alg_metric, "alg_display": alg_meta["display_name"],
                "baseline": baseline_metric, "baseline_display": baseline_meta["display_name"],
                "n": len(common_index), "alg_mean_f1": alg.loc[common_index].mean(),
                "baseline_mean_f1": baseline.loc[common_index].mean(),
                "mean_delta_f1": delta.mean(), "ci_low": ci_low, "ci_high": ci_high,
                "wins": int((delta > 1e-12).sum()), "ties": int((np.abs(delta) <= 1e-12).sum()),
                "losses": int((delta < -1e-12).sum()), "wilcoxon_p_raw": p_value,
            })
        adjusted = holm_adjust(np.asarray(raw_p_values))
        for record, adjusted_value in zip(block, adjusted):
            record["wilcoxon_p_holm"] = adjusted_value
        paired_records.extend(block)
    paired = pd.DataFrame(paired_records)
    paired.to_csv(DATA / "paired_two_alg_vs_40_baselines_test186.csv", index=False)

    efficiency = pd.read_csv(AGGREGATE / "baseline_efficiency_test183.csv.gz")
    runtime_records: list[dict[str, object]] = []
    for metric, part in efficiency.groupby("metric"):
        n_timed = part["n_timed_pairs"].sum()
        if n_timed > 0:
            runtime_records.append({
                "metric": metric, "method_type": "baseline_measured",
                "ms_per_pair": 1000.0 * part["total_metric_sec"].sum() / n_timed,
                "n_timed_pairs": n_timed,
            })
    framework_efficiency = pd.read_csv(AGGREGATE / "framework_configuration_efficiency.csv")
    selected_efficiency = framework_efficiency.loc[framework_efficiency["metric"].eq(ALG_STAR)].iloc[0]
    selected_ms = 1000.0 * selected_efficiency["estimated_metric_compute_sec_per_pair"]
    runtime = pd.concat([pd.DataFrame([{
        "metric": ALG_STAR, "method_type": "ALG_component_estimate",
        "ms_per_pair": selected_ms, "n_timed_pairs": np.nan,
    }]), pd.DataFrame(runtime_records)], ignore_index=True)
    runtime["relative_to_selected_alg"] = runtime["ms_per_pair"] / selected_ms
    runtime = runtime.merge(roles[["metric", "display_name"]], on="metric", how="left")
    runtime.to_csv(DATA / "runtime_all_methods.csv", index=False)

    # Controlled sensitivity, now including the actually evaluated ALG* column.
    scenario_labels = {
        "uniform_translation": "Translation", "nested_support": "Nested support",
        "nonconvex_disk": "Non-convex disk", "nonconvex_ring": "Non-convex ring",
        "nonconvex_checkerboard": "Checkerboard",
    }
    raw = pd.read_csv(CONTROLLED_RAW)
    sensitivity_records: list[dict[str, object]] = []
    for metric in included["metric"]:
        value_column = metric if metric == ALG_STAR else f"baseline_{metric}"
        if value_column not in raw.columns:
            continue
        display_name = included.loc[included["metric"].eq(metric), "display_name"].iloc[0]
        for scenario, label in scenario_labels.items():
            part = raw.loc[raw["scenario"].eq(scenario), ["level", value_column]].dropna().copy()
            if scenario == "nested_support":
                part["severity"] = np.abs(np.log(part["level"].astype(float)))
            elif scenario == "uniform_translation":
                part["severity"] = np.abs(part["level"].astype(float))
            else:
                part["severity"] = part["level"].astype(float)
            curve = part.groupby("severity", as_index=False)[value_column].mean()
            rho = spearmanr(curve["severity"], curve[value_column]).statistic
            sensitivity_records.append({
                "metric": metric, "display_name": display_name, "scenario": scenario,
                "scenario_label": label,
                "oriented_spearman": 0.0 if not np.isfinite(rho) else float(-rho),
                "n_levels": len(curve),
            })
    sensitivity = pd.DataFrame(sensitivity_records)
    sensitivity.to_csv(DATA / "controlled_sensitivity_all_baselines.csv", index=False)

    result_summary = (leaderboard.loc[leaderboard["metric"].isin([ALG_STAR, INITIAL_ALG, "energy_similarity"])]
                      .merge(included[["metric", "display_name"]], on="metric", how="left")
                      .sort_values("mean_f1", ascending=False))
    result_summary.to_csv(TABLES / "article_result_summary.csv", index=False)
    paired.loc[paired["baseline"].eq("energy_similarity")].to_csv(TABLES / "article_paired_energy.csv", index=False)
    support_summary.loc[support_summary["metric"].isin([ALG_STAR, INITIAL_ALG])].to_csv(
        TABLES / "article_support_summary.csv", index=False
    )

    exploratory = pd.read_csv(AGGREGATE / "framework_exploratory_test_ranking.csv")
    neural_ranking = pd.read_csv(NEURAL / "framework_all_168960_neural_ranking.csv", usecols=["metric", "mean_f1", "median_f1"])
    combined_ranking = exploratory[["metric", "mean_f1", "median_f1"]].merge(
        neural_ranking, on="metric", suffixes=("_183", "_neural"), validate="one_to_one"
    )
    combined_ranking["mean_f1"] = (183 * combined_ranking["mean_f1_183"] + 3 * combined_ranking["mean_f1_neural"]) / 186
    # Median cannot be reconstructed from two medians; retain benchmark median and label it explicitly in the CSV.
    combined_ranking["median_f1"] = combined_ranking["median_f1_183"]
    combined_ranking.to_csv(DATA / "framework_test186_ranking.csv", index=False)

    ablation_metrics = {
        "Initial ALG - local only": "overlap_cosine_sym",
        "Initial ALG - global only": "cosine_pooled",
        "Initial ALG - local-global": INITIAL_ALG,
        "ALG* - local only": ALG_STAR.replace("lambda0.7", "lambda1.0"),
        "ALG* - global only": ALG_STAR.replace("lambda0.7", "lambda0.0"),
        "ALG* - local-global": ALG_STAR,
    }
    ablation_baseline_metrics = {
        "overlap_cosine_sym", "cosine_pooled", INITIAL_ALG,
    }
    ablation_baseline_rows = pd.concat([
        baselines_all.loc[
            baselines_all["metric"].isin(ablation_baseline_metrics),
            ["dataset", "metric", "f1"],
        ],
        neural.loc[
            neural["metric"].isin(ablation_baseline_metrics),
            ["dataset", "metric", "f1"],
        ],
    ], ignore_index=True)
    baseline_lookup = (ablation_baseline_rows.groupby("metric")
                       .agg(mean_f1=("f1", "mean"), median_f1=("f1", "median")))
    framework_lookup = combined_ranking.set_index("metric")
    ablation_records = []
    for label, metric in ablation_metrics.items():
        row = framework_lookup.loc[metric] if metric.startswith("algfw_") else baseline_lookup.loc[metric]
        ablation_records.append({"variant": label, "metric": metric, "mean_f1": row["mean_f1"], "median_f1": row["median_f1"]})
    ablation = pd.DataFrame(ablation_records)
    ablation.to_csv(TABLES / "article_ablation.csv", index=False)

    return {
        "primary_dataset_results": primary,
        "method_by_support": support_summary,
        "complete_comparator_suite": comparator,
        "common_all": common,
        "paired_comparisons": paired,
        "runtime": runtime,
        "controlled_sensitivity": sensitivity,
        "article_result_summary": result_summary,
        "article_ablation": ablation,
    }

def configure_style() -> None:
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    })


def save(fig: plt.Figure, basename: str, dpi: int = 260) -> None:
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / f"{basename}.png", dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def clean_label(value: str) -> str:
    return value.replace("development-selected", "training-selected").replace("_", " ")


def method_color(metric: str) -> str:
    if metric == ALG_STAR:
        return PURPLE
    if metric == INITIAL_ALG:
        return ORANGE
    return BLUE


def original_domain(metric: str) -> str:
    if metric.startswith("algfw_") or metric == INITIAL_ALG:
        return "ALG"
    if metric in GENERATIVE:
        return "Generative-model evaluation"
    if metric in OPTIMAL_TRANSPORT:
        return "Optimal transport"
    if metric in POINT_SET:
        return "Point-set / 3D"
    if metric == "best_buddies_similarity":
        return "Image matching"
    if metric == "local_scaling_affinity":
        return "Spectral clustering"
    if metric == "persistence_bottleneck_similarity":
        return "Topology"
    return "General-purpose"


def add_panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(0.0, 1.015, label, transform=ax.transAxes, ha="left", va="bottom", fontweight="bold")


def figure_1_motivation() -> None:
    def local_agreement(a: np.ndarray, b: np.ndarray) -> float:
        def directed(x: np.ndarray, y: np.ndarray) -> float:
            internal = np.linalg.norm(x[:, None, :] - x[None, :, :], axis=2)
            np.fill_diagonal(internal, np.inf)
            radius = internal.min(axis=1)
            cross = np.linalg.norm(x[:, None, :] - y[None, :, :], axis=2).min(axis=1)
            return float(np.mean(cross <= radius + 1e-12))
        return 0.5 * (directed(a, b) + directed(b, a))

    a = np.array([[-2.2, -0.5], [-1.7, 0.5], [-1.1, -0.3], [1.1, 0.3], [1.7, -0.5], [2.2, 0.5]])
    jitter = np.array([[.05, .03], [-.04, .02], [.02, -.05], [-.02, .05], [.04, -.02], [-.05, -.03]])
    cases = [
        a + jitter,
        np.array([[-.35, -2.0], [.35, -1.2], [-.3, -.5], [.3, .5], [-.35, 1.2], [.35, 2.0]]),
        a + np.array([0.7, 0.0]),
        np.array([[1.8, 1.7], [2.5, 1.5], [3.0, 2.2], [2.2, 2.8], [3.2, 3.0], [3.7, 2.4]]),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(13.2, 3.0), constrained_layout=True)
    for index, (ax, b) in enumerate(zip(axes, cases)):
        local = local_agreement(a, b)
        glob = 1.0 / (1.0 + float(np.linalg.norm(a.mean(axis=0) - b.mean(axis=0))))
        ax.scatter(a[:, 0], a[:, 1], s=52, color=PURPLE, label="Set A", zorder=3)
        ax.scatter(b[:, 0], b[:, 1], s=58, color="#168A31", marker="x", linewidth=2.2, label="Set B", zorder=3)
        ax.scatter(*a.mean(axis=0), s=130, marker="*", facecolor="white", edgecolor=PURPLE, linewidth=1.8, zorder=4)
        ax.scatter(*b.mean(axis=0), s=130, marker="*", facecolor="white", edgecolor="#168A31", linewidth=1.8, zorder=4)
        add_panel_label(ax, f"({chr(97 + index)})")
        ax.text(.5, .025, f"$L={local:.2f}$   $G={glob:.2f}$", transform=ax.transAxes,
                ha="center", va="bottom", fontsize=9,
                bbox=dict(facecolor="white", edgecolor="#D1D5DB", pad=2.5))
        ax.set_xlim(-3.0, 4.2); ax.set_ylim(-3.0, 3.6)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True); spine.set_color("#B8B8B8")
    axes[0].legend(loc="upper left", frameon=False, fontsize=8)
    save(fig, "alg_local_global_motivation", dpi=300)


def figure_2_leaderboard() -> None:
    scores = pd.read_csv(DATA / "confirmatory_selected_alg_plus_baselines_test186.csv")
    names = pd.read_csv(METADATA / "method_roles_40baselines.csv")[["metric", "display_name"]]
    scores = scores.merge(names, on="metric", how="left")
    scores = scores[scores["n_datasets"] == scores["n_datasets"].max()].sort_values("mean_f1")
    scores["display_name"] = scores["display_name"].fillna(scores["metric"]).map(clean_label)
    y = np.arange(len(scores))
    colors = [method_color(metric) for metric in scores["metric"]]
    fig, axes = plt.subplots(1, 2, figsize=(14.5, 15.5), sharey=True,
                             gridspec_kw={"width_ratios": [1.55, 1.0], "wspace": 0.08})
    axes[0].hlines(y, scores["mean_f1"].min() - .003, scores["mean_f1"], color=GRID, linewidth=.8)
    axes[0].scatter(scores["mean_f1"], y, c=colors, s=33, zorder=3)
    axes[0].set_yticks(y, scores["display_name"])
    axes[0].set_xlabel("Dataset-macro mean F1-score (186 held-out datasets)")
    axes[0].set_xlim(scores["mean_f1"].min() - .004, scores["mean_f1"].max() + .004)
    axes[1].hlines(y, 0, scores["n_dataset_wins"] / 186, color=GRID, linewidth=.8)
    axes[1].scatter(scores["n_dataset_wins"] / 186, y, c=colors, s=33, zorder=3)
    axes[1].set_xlabel("Fractional dataset win rate")
    axes[1].xaxis.set_major_formatter(PercentFormatter(1.0))
    for ax, label in zip(axes, ["(a)", "(b)"]):
        add_panel_label(ax, label); ax.grid(axis="x", alpha=.18)
    handles = [Line2D([0], [0], marker="o", color="none", markerfacecolor=c, markeredgecolor=c, label=l)
               for c, l in [(PURPLE, "ALG*"), (ORANGE, "Initial ALG"), (BLUE, "Baselines")]]
    fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(.58, .997))
    fig.subplots_adjust(left=.34, right=.98, top=.965, bottom=.05)
    save(fig, "01_confirmatory_leaderboard")


def figure_3_paired() -> None:
    paired = pd.read_csv(DATA / "paired_two_alg_vs_40_baselines_test186.csv")
    order = (paired.groupby(["baseline", "baseline_display"], as_index=False)["baseline_mean_f1"]
             .mean().sort_values("baseline_mean_f1")["baseline"].tolist())
    labels = (paired.drop_duplicates("baseline").set_index("baseline")["baseline_display"]
              .reindex(order).map(clean_label))
    positions = np.arange(len(order))
    fig, axes = plt.subplots(1, 2, figsize=(15.0, 15.5), sharey=True,
                             gridspec_kw={"width_ratios": [1.35, 1.0], "wspace": .08})
    styles = [(ALG_STAR, "ALG*", PURPLE, -.14), (INITIAL_ALG, "Initial ALG", ORANGE, .14)]
    for alg_metric, label, color, offset in styles:
        part = paired[paired["alg_metric"] == alg_metric].set_index("baseline").reindex(order)
        y = positions + offset
        mean = part["mean_delta_f1"].to_numpy()
        xerr = np.vstack([mean - part["ci_low"].to_numpy(), part["ci_high"].to_numpy() - mean])
        axes[0].errorbar(mean, y, xerr=xerr, fmt="o", color=color, markersize=4,
                         linewidth=.9, capsize=2, label=label)
        axes[1].scatter(part["wins"].to_numpy() / part["n"].to_numpy(), y,
                        color=color, s=24, label=label)
    axes[0].axvline(0, color="black", linestyle="--", linewidth=.8)
    axes[0].set_xlabel("Paired mean F1 difference (ALG - baseline), 95% bootstrap CI")
    axes[1].axvline(.5, color="black", linestyle="--", linewidth=.8)
    axes[1].set_xlabel("Fraction of datasets won by ALG")
    axes[1].xaxis.set_major_formatter(PercentFormatter(1.0))
    axes[0].set_yticks(positions, labels)
    for ax, label in zip(axes, ["(a)", "(b)"]):
        add_panel_label(ax, label); ax.grid(axis="x", alpha=.18)
    axes[0].legend(loc="lower right", frameon=False)
    fig.subplots_adjust(left=.34, right=.98, top=.98, bottom=.05)
    save(fig, "05_paired_effects")

from matplotlib.colors import LinearSegmentedColormap

def figure_4_family_heatmap() -> None:
    frame = pd.read_csv(DATA / "all_primary_methods_by_support_test186_40baselines.csv")
    frame["domain"] = frame["metric"].map(original_domain)

    display = frame["display_name"].map(clean_label).copy()
    display.loc[frame["metric"] == "gaussian_frechet_similarity"] += "†"

    columns = [
        ("overall_mean_f1", "Overall"),
        ("graphs_mean_f1", "Graphs\n(n=15)"),
        ("images_mean_f1", "Images\n(n=12)"),
        ("openml_mean_f1", "OpenML\n(n=57)"),
        ("tabular_mean_f1", "Tabular\n(n=1)"),
        ("text_mean_f1", "Text\n(n=6)"),
        ("time_series_mean_f1", "Time series\n(n=92)"),
        ("neural_models_mean_f1", "Neural models\n(n=3)"),
    ]

    matrix = frame[[c for c, _ in columns]].to_numpy(float)
    rounded = np.round(matrix, 3)

    # White -> green colormap, with 1.000 at the strongest green
    cmap = LinearSegmentedColormap.from_list(
        "white_to_green",
        ["#ffffff", "#edf8e9", "#bae4b3", "#74c476", "#31a354", "#006d2c"]
    )

    fig, ax = plt.subplots(figsize=(14.8, 19.0))

    image = ax.imshow(
        matrix,
        aspect="auto",
        cmap=cmap,
        vmin=np.nanmin(matrix),
        vmax=1.0,
    )

    ax.set_xticks(
        np.arange(len(columns)),
        [label for _, label in columns],
        fontsize=11,
    )
    ax.xaxis.tick_top()

    ax.set_yticks(np.arange(len(frame)))
    ax.set_yticklabels(display, fontsize=9)

    # Color method names according to their original application/domain
    for tick, domain in zip(ax.get_yticklabels(), frame["domain"]):
        if domain in DOMAIN_COLORS:
            tick.set_color(DOMAIN_COLORS[domain])

    # Identify best and second-best distinct rounded values per column
    best_rows_by_col = {}
    second_rows_by_col = {}

    for col in range(rounded.shape[1]):
        values = rounded[:, col]
        finite_values = values[~np.isnan(values)]

        if len(finite_values) == 0:
            best_rows_by_col[col] = set()
            second_rows_by_col[col] = set()
            continue

        distinct_desc = sorted(set(finite_values.tolist()), reverse=True)

        best_value = distinct_desc[0]
        second_value = distinct_desc[1] if len(distinct_desc) > 1 else None

        best_rows = set(np.where(values == best_value)[0].tolist())
        second_rows = (
            set(np.where(values == second_value)[0].tolist())
            if second_value is not None else set()
        )

        best_rows_by_col[col] = best_rows
        second_rows_by_col[col] = second_rows

    # Display values inside cells
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = matrix[row, col]

            if np.isnan(value):
                text = "—"
                text_color = "black"
                fontweight = "normal"
            else:
                text = f"{value:.3f}"

                # Dark text on light cells, white on darker green cells
                text_color = "black" if value < 0.78 else "white"

                if row in best_rows_by_col[col]:
                    fontweight = "bold"
                else:
                    fontweight = "normal"

            ax.text(
                col,
                row,
                text,
                ha="center",
                va="center",
                fontsize=6.7,
                color=text_color,
                fontweight=fontweight,
            )

            # Underline the second-best distinct value(s) in each column
            if (not np.isnan(value)) and (row in second_rows_by_col[col]):
                underline_y = row + 0.22
                ax.plot(
                    [col - 0.22, col + 0.22],
                    [underline_y, underline_y],
                    color=text_color,
                    linewidth=1.1,
                    solid_capstyle="round",
                )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        fraction=.022,
        pad=.025,
    )
    colorbar.set_label("Mean F1-score")

    present_domains = [
        name
        for name in DOMAIN_COLORS
        if name in set(frame["domain"].dropna())
    ]

    handles = [
        Patch(
            facecolor=DOMAIN_COLORS[name],
            edgecolor="none",
            label=name,
        )
        for name in present_domains
    ]

    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(.60, .950),
        ncol=4,
        frameon=False,
        fontsize=8.7,
        columnspacing=1.25,
        handlelength=1.4,
    )

    ax.tick_params(axis="x", length=0)
    ax.tick_params(axis="y", length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.subplots_adjust(
        left=.34,
        right=.93,
        top=.90,
        bottom=.035,
    )

    save(fig, "03_category_performance")
def figure_5_stability() -> None:
    development = pd.read_csv(AGGREGATE / "framework_development_ranking.csv", usecols=["metric", "mean_f1"])
    test = pd.read_csv(DATA / "framework_test186_ranking.csv", usecols=["metric", "mean_f1"])
    merged = development.rename(columns={"mean_f1": "training"}).merge(
        test.rename(columns={"mean_f1": "test"}), on="metric", validate="one_to_one")
    rho = spearmanr(merged["training"], merged["test"]).statistic
    selected = merged.loc[merged["metric"] == ALG_STAR].iloc[0]
    oracle = merged.sort_values("test", ascending=False).iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.4), gridspec_kw={"width_ratios": [1.15, 1]})
    hb = axes[0].hexbin(merged["training"], merged["test"], gridsize=65, mincnt=1,
                        cmap="Blues", bins="log", rasterized=True)
    limits = [merged[["training", "test"]].min().min() - .004,
              merged[["training", "test"]].max().max() + .004]
    axes[0].plot(limits, limits, linestyle="--", color=".5", linewidth=.8)
    axes[0].scatter(selected["training"], selected["test"], marker="*", s=150,
                    color=PURPLE, edgecolor="white", linewidth=.8, label="ALG*", zorder=5)
    axes[0].scatter(oracle["training"], oracle["test"], marker="D", s=60,
                    color=ORANGE, edgecolor="white", linewidth=.8, label="Retrospective test optimum", zorder=5)
    axes[0].set_xlabel("Training mean F1-score (60 datasets)")
    axes[0].set_ylabel("Held-out test mean F1-score (186 datasets)")
    axes[0].text(.03, .96, rf"Spearman $\rho={rho:.3f}$", transform=axes[0].transAxes, va="top")
    axes[0].legend(frameon=False, loc="lower right")
    fig.colorbar(hb, ax=axes[0], label="Log count")
    ordered = development.sort_values("mean_f1", ascending=False)
    regret = ordered["mean_f1"].max() - ordered["mean_f1"].to_numpy()
    axes[1].plot(np.sort(regret), np.arange(1, len(regret) + 1) / len(regret), color=PURPLE, linewidth=2)
    axes[1].set_xlabel(r"Training regret $F_1^* - F_1(\theta)$")
    axes[1].set_ylabel("Cumulative fraction of configurations")
    axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
    for ax, label in zip(axes, ["(a)", "(b)"]): add_panel_label(ax, label); ax.grid(alpha=.18)
    fig.tight_layout()
    save(fig, "02_development_test_generalization")


def figure_6_hyperparameters() -> None:
    dev = pd.read_csv(AGGREGATE / "framework_development_ranking.csv")
    dev = dev.sort_values("mean_f1", ascending=False)
    panels = [
        ("local_family", "Local similarity"), ("local_geometry", "Local geometry"),
        ("global_family", "Global discrepancy"), ("lambda", r"Fusion weight $\lambda$"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(12.8, 8.8))
    for index, (ax, (column, ylabel)) in enumerate(zip(axes.ravel(), panels)):
        frames = []
        for top_n, name in [(1000, "Top 1000"), (100, "Top 100")]:
            share = dev.head(top_n)[column].fillna("not applicable").astype(str).value_counts(normalize=True)
            frames.append(share.rename(name))
        table = pd.concat(frames, axis=1).fillna(0).sort_values("Top 1000")
        y = np.arange(len(table)); height = .36
        ax.barh(y - height / 2, table["Top 1000"], height, color=BLUE, label="Top 1000")
        ax.barh(y + height / 2, table["Top 100"], height, color=PURPLE, label="Top 100")
        ax.set_yticks(y, [v.replace("_", " ") for v in table.index])
        ax.set_ylabel(ylabel); ax.set_xlabel("Share of configurations")
        ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.grid(axis="x", alpha=.18)
        add_panel_label(ax, f"({chr(97 + index)})")
    axes[0, 0].legend(frameon=False, ncol=2, loc="lower right")
    fig.tight_layout()
    save(fig, "04_hyperparameter_prevalence")


def _normalize(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    low, high = np.nanmin(values), np.nanmax(values)
    return np.ones_like(values) if np.isclose(low, high) else (values - low) / (high - low)


def figure_7_controlled_curves() -> None:
    columns = {
        "Initial ALG": "baseline_overlap_cosine_plus_pooled_cosine",
        "Energy distance": "baseline_energy_similarity",
        "Local scaling": "baseline_local_scaling_affinity",
        "MMD-RBF": "baseline_mmd_rbf_similarity",
        "Coverage": "baseline_coverage_sym_k5",
    }
    usecols = ["experiment", "case", "level"]
    for column in columns.values(): usecols.extend([column, f"{column}_se"])
    data = pd.read_csv(SYNTHETIC, usecols=usecols)
    selections = [
        ((data.experiment == "translation_outlier") & (data.case == "none"), r"Translation magnitude $|\mu|$", "absolute"),
        (data.experiment == "mode_sequential", "Number of modes removed", "removed"),
        (data.experiment == "mode_simultaneous", "Dominant-mode probability", "identity"),
    ]
    palette = {"Initial ALG": PURPLE, "Energy distance": ORANGE, "Local scaling": "#2E8B57",
               "MMD-RBF": BLUE, "Coverage": "#7F8C8D"}
    fig, axes = plt.subplots(1, 3, figsize=(13.4, 4.0), sharey=True)
    for index, (ax, (mask, xlabel, transform)) in enumerate(zip(axes, selections)):
        sub = data.loc[mask].copy()
        if transform == "absolute":
            sub["severity"] = sub["level"].abs()
        elif transform == "removed":
            sub["severity"] = sub["level"].max() - sub["level"]
        else:
            sub["severity"] = sub["level"]
        for name, column in columns.items():
            curve = sub.groupby("severity", as_index=False)[[column, f"{column}_se"]].mean().sort_values("severity")
            x = curve["severity"].to_numpy(float); y = _normalize(curve[column].to_numpy(float))
            se_raw = curve[f"{column}_se"].to_numpy(float)
            scale = max(np.nanmax(curve[column]) - np.nanmin(curve[column]), 1e-12)
            se = se_raw / scale
            ax.plot(x, y, color=palette[name], linewidth=2.3 if name == "Initial ALG" else 1.6, label=name)
            ax.fill_between(x, np.clip(y - se, 0, 1), np.clip(y + se, 0, 1), color=palette[name], alpha=.09)
        ax.set_xlabel(xlabel); ax.grid(alpha=.18); add_panel_label(ax, f"({chr(97 + index)})")
    axes[0].set_ylabel("Within-method normalized similarity")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=5, frameon=False, bbox_to_anchor=(.5, 1.02))
    fig.tight_layout(rect=(0, 0, 1, .92))
    save(fig, "07_controlled_perturbations")


from matplotlib.colors import LinearSegmentedColormap
def figure_8_controlled_all() -> None:
    result = pd.read_csv(
        DATA / "controlled_sensitivity_all_baselines.csv"
    )

    method_labels = (
        result[["metric", "display_name"]]
        .drop_duplicates("metric")
        .set_index("metric")["display_name"]
    )

    matrix = result.pivot(
        index="metric",
        columns="scenario_label",
        values="oriented_spearman",
    )

    order = [
        "Translation",
        "Nested support",
        "Non-convex disk",
        "Non-convex ring",
        "Checkerboard",
    ]

    matrix = (
        matrix[order]
        .assign(_mean=lambda frame: frame.mean(axis=1))
        .sort_values("_mean")
        .drop(columns="_mean")
    )

    # Les classements sont calculés à partir des valeurs affichées,
    # donc après arrondi à trois décimales.
    rounded = np.round(matrix.to_numpy(dtype=float), 3)

    best_by_column: dict[int, float] = {}
    second_by_column: dict[int, float | None] = {}

    for column in range(rounded.shape[1]):
        distinct = np.sort(
            np.unique(rounded[:, column])
        )[::-1]

        best_by_column[column] = float(distinct[0])
        second_by_column[column] = (
            float(distinct[1])
            if len(distinct) > 1
            else None
        )

    observed_minimum = float(
        np.nanmin(matrix.to_numpy(dtype=float))
    )

    white_green = LinearSegmentedColormap.from_list(
        "controlled_white_green",
        ["#FFFFFF", "#006D2C"],
    )

    fig, ax = plt.subplots(figsize=(9.8, 15.0))

    image = ax.imshow(
        matrix.to_numpy(),
        aspect="auto",
        cmap=white_green,
        norm=Normalize(
            vmin=observed_minimum,
            vmax=1.0,
        ),
    )

    ax.set_xticks(
        np.arange(matrix.shape[1]),
        matrix.columns,
        rotation=28,
        ha="right",
    )

    labels = [
        clean_label(method_labels.loc[metric])
        for metric in matrix.index
    ]

    ax.set_yticks(
        np.arange(matrix.shape[0]),
        labels,
        fontsize=8,
    )

    # ALG* est mis en gras uniquement s'il est réellement présent.
    if ALG_STAR in matrix.index:
        alg_star_row = matrix.index.get_loc(ALG_STAR)
        ax.get_yticklabels()[alg_star_row].set_fontweight(
            "bold"
        )

    ax.set_xlabel("Controlled perturbation")
    ax.set_ylabel("Method")

    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = float(rounded[row, column])

            normalized = (
                matrix.iloc[row, column] - observed_minimum
            ) / max(1.0 - observed_minimum, 1e-12)

            text_color = (
                "white"
                if normalized >= 0.58
                else "black"
            )

            is_best = (
                value == best_by_column[column]
            )

            is_second = (
                second_by_column[column] is not None
                and value == second_by_column[column]
            )

            ax.text(
                column,
                row,
                f"{value:.3f}",
                ha="center",
                va="center",
                fontsize=6.2,
                color=text_color,
                fontweight=(
                    "bold" if is_best else "normal"
                ),
            )

            # Matplotlib ne fournit pas un soulignement fiable
            # pour les nombres. Le trait est donc dessiné sous
            # toutes les valeurs ex aequo au deuxième score distinct.
            if is_second:
                ax.plot(
                    [
                        column - 0.23,
                        column + 0.23,
                    ],
                    [
                        row + 0.19,
                        row + 0.19,
                    ],
                    color=text_color,
                    linewidth=0.65,
                    solid_capstyle="butt",
                )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.025,
        pad=0.025,
    )
    colorbar.set_label(
        "Oriented Spearman response"
    )

    fig.tight_layout()
    save(
        fig,
        "08_controlled_all_baselines",
    )

def figure_9_runtime() -> None:
    runtime = pd.read_csv(DATA / "runtime_all_methods.csv")
    roles = pd.read_csv(METADATA / "method_roles_40baselines.csv")[["metric", "display_name"]]
    runtime = runtime.merge(roles, on="metric", how="left", suffixes=("", "_role"))
    runtime["label"] = runtime["display_name_role"].fillna(runtime["display_name"]).fillna(runtime["metric"]).map(clean_label)
    runtime = runtime.dropna(subset=["ms_per_pair"]).sort_values("ms_per_pair", ascending=True)
    y = np.arange(len(runtime)); colors = [method_color(metric) for metric in runtime["metric"]]
    fig, ax = plt.subplots(figsize=(11.0, 13.5))
    ax.barh(y, runtime["ms_per_pair"], color=colors, height=.72)
    ax.set_yticks(y, runtime["label"], fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel("Milliseconds per embedding-set pair (log scale)")
    ax.grid(axis="x", alpha=.18)
    handles = [Patch(facecolor=c, label=l) for c, l in [(PURPLE, "ALG*"), (ORANGE, "Initial ALG"), (BLUE, "Baselines")]]
    ax.legend(handles=handles, frameon=False, loc="lower right")
    fig.tight_layout()
    save(fig, "06_runtime_comparison")


def figure_10_common133() -> None:
    frame = pd.read_csv(DATA / "common_all_42_methods.csv").sort_values("mean_f1_common")
    frame["display_name"] = frame["display_name"].map(clean_label)
    y = np.arange(len(frame))
    colors = []
    for metric in frame["metric"]:
        if metric == ALG_STAR: colors.append(PURPLE)
        elif metric == INITIAL_ALG: colors.append(ORANGE)
        elif metric == "gaussian_frechet_similarity": colors.append("#2E8B57")
        else: colors.append(BLUE)
    fig, ax = plt.subplots(figsize=(10.5, 14.0))
    lower = frame["mean_f1_common"].min() - .003
    ax.hlines(y, lower, frame["mean_f1_common"], color=GRID, linewidth=.8)
    ax.scatter(frame["mean_f1_common"], y, c=colors, s=34, zorder=3)
    ax.set_yticks(y, frame["display_name"], fontsize=8.5)
    ax.set_xlabel("Mean F1-score on the common 136-dataset subset")
    ax.set_xlim(lower, frame["mean_f1_common"].max() + .004); ax.grid(axis="x", alpha=.18)
    handles = [Patch(facecolor=c, label=l) for c, l in [
        (PURPLE, "ALG*"), (ORANGE, "Initial ALG"), ("#2E8B57", "Gaussian Fréchet"), (BLUE, "Other baselines")]]
    ax.legend(handles=handles, frameon=False, loc="lower right")
    fig.tight_layout()
    save(fig, "07_common136_all_methods")


def figure_11_neural_all_methods() -> None:
    """All methods on the three held-out neural-representation datasets."""
    data = pd.read_csv(NEURAL / "all_methods_neural_per_dataset.csv")
    roles = pd.read_csv(METADATA / "method_roles_40baselines.csv")[["metric", "display_name"]]
    labels = roles.drop_duplicates("metric").set_index("metric")["display_name"].to_dict()
    dataset_order = [
        "MNIST_mlp_activation_clouds_external",
        "KMNIST_mlp_activation_clouds_external",
        "USPS_mlp_activation_clouds_external",
    ]
    matrix = data.pivot(index="metric", columns="dataset", values="f1")[dataset_order]
    matrix["Mean"] = matrix.mean(axis=1)
    matrix = matrix.sort_values("Mean", ascending=True)
    rounded = np.round(matrix.to_numpy(float), 3)
    observed_minimum = float(np.nanmin(matrix.to_numpy(float)))
    white_green = LinearSegmentedColormap.from_list("neural_white_green", ["#FFFFFF", "#006D2C"])
    fig, ax = plt.subplots(figsize=(9.2, 15.5))
    image = ax.imshow(matrix.to_numpy(float), aspect="auto", cmap=white_green,
                      norm=Normalize(vmin=observed_minimum, vmax=1.0))
    ax.set_xticks(np.arange(4), ["MNIST", "KMNIST", "USPS", "Mean"])
    display_labels = [clean_label(labels.get(metric, metric)) for metric in matrix.index]
    ax.set_yticks(np.arange(len(matrix)), display_labels, fontsize=8)
    if ALG_STAR in matrix.index:
        ax.get_yticklabels()[matrix.index.get_loc(ALG_STAR)].set_fontweight("bold")
    for column in range(rounded.shape[1]):
        distinct = np.sort(np.unique(rounded[np.isfinite(rounded[:, column]), column]))[::-1]
        best = distinct[0]
        second = distinct[1] if len(distinct) > 1 else None
        for row in range(rounded.shape[0]):
            value = float(rounded[row, column])
            raw = float(matrix.iloc[row, column])
            normalized = (raw - observed_minimum) / max(1.0 - observed_minimum, 1e-12)
            color = "white" if normalized >= .58 else "black"
            ax.text(column, row, f"{value:.3f}", ha="center", va="center", fontsize=6.6,
                    color=color, fontweight="bold" if value == best else "normal")
            if second is not None and value == second:
                ax.plot([column - .23, column + .23], [row + .19, row + .19],
                        color=color, linewidth=.65, solid_capstyle="butt")
    ax.set_xlabel("Neural-representation dataset")
    ax.set_ylabel("Method")
    colorbar = fig.colorbar(image, ax=ax, fraction=.028, pad=.025)
    colorbar.set_label("F1-score")
    fig.tight_layout()
    save(fig, "09_neural_all_methods")

def generate_all() -> list[Path]:
    configure_style()
    figure_1_motivation()
    figure_2_leaderboard()
    figure_3_paired()
    figure_4_family_heatmap()
    figure_5_stability()
    figure_6_hyperparameters()
    figure_7_controlled_curves()
    figure_8_controlled_all()
    figure_9_runtime()
    figure_10_common133()
    figure_11_neural_all_methods()
    return [
        FIGURES / "alg_local_global_motivation.png",
        FIGURES / "01_confirmatory_leaderboard.png",
        FIGURES / "05_paired_effects.png",
        FIGURES / "03_category_performance.png",
        FIGURES / "02_development_test_generalization.png",
        FIGURES / "04_hyperparameter_prevalence.png",
        FIGURES / "07_controlled_perturbations.png",
        FIGURES / "08_controlled_all_baselines.png",
        FIGURES / "06_runtime_comparison.png",
        FIGURES / "07_common136_all_methods.png",
        FIGURES / "09_neural_all_methods.png",
    ]



## 2. Input audit

In [ ]:
required_inputs = [
    AGGREGATE / "campaign_summary.json",
    AGGREGATE / "baseline_results_test183.csv.gz",
    AGGREGATE / "baseline_efficiency_test183.csv.gz",
    AGGREGATE / "frozen_global_selection_test183_all_metrics.csv",
    AGGREGATE / "framework_development_ranking.csv",
    AGGREGATE / "framework_exploratory_test_ranking.csv",
    AGGREGATE / "framework_configuration_efficiency.csv",
    NEURAL / "all_methods_neural_per_dataset.csv",
    NEURAL / "framework_all_168960_neural_ranking.csv",
    CONTROLLED_RAW,
    SYNTHETIC,
    METADATA / "method_roles_40baselines.csv",
]
missing = [path for path in required_inputs if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing reproducibility inputs:\n" + "\n".join(map(str, missing)))
input_audit = pd.DataFrame({
    "path": [str(path.relative_to(PROJECT)) for path in required_inputs],
    "size_MiB": [path.stat().st_size / 2**20 for path in required_inputs],
})
display(input_audit)


## 3. Recompute all numerical results and manuscript tables

In [ ]:
BOOTSTRAP_REPETITIONS = 10_000
RANDOM_SEED = 20260925

result_tables = rebuild_article_results(
    bootstrap_repetitions=BOOTSTRAP_REPETITIONS,
    seed=RANDOM_SEED,
)

actual_shapes = {name: frame.shape for name, frame in result_tables.items()}
assert result_tables["primary_dataset_results"]["dataset"].nunique() == 186
assert result_tables["method_by_support"].shape[0] == 42
assert result_tables["controlled_sensitivity"]["metric"].eq(ALG_STAR).any()
display(pd.DataFrame([
    {"table": name, "rows": shape[0], "columns": shape[1]}
    for name, shape in actual_shapes.items()
]))


### Principal manuscript result table

In [ ]:
display(result_tables["article_result_summary"].reset_index(drop=True))

### Ablation table

In [ ]:
display(result_tables["article_ablation"].reset_index(drop=True))

### Complete 42-method comparator table

In [ ]:
display(result_tables["complete_comparator_suite"][[
    "rank", "display_name", "role", "overall_n", "overall_mean_f1", "family"
]].reset_index(drop=True))

### All 80 paired comparisons: two ALG variants × 40 baselines

In [ ]:
display(result_tables["paired_comparisons"].sort_values(["alg_display", "baseline_mean_f1"], ascending=[True, False]).reset_index(drop=True))

### Cross-support result matrix for all 42 methods

In [ ]:
display(result_tables["method_by_support"].reset_index(drop=True))

### Common-133 comparison for all 42 methods

In [ ]:
display(result_tables["common_all"].reset_index(drop=True))

### Efficiency table for every timed method

In [ ]:
display(result_tables["runtime"].sort_values("ms_per_pair").reset_index(drop=True))

### Controlled sensitivity: all methods and all perturbations

In [ ]:
display(result_tables["controlled_sensitivity"].reset_index(drop=True))

## 4. Regenerate every manuscript figure

In [ ]:
figure_paths = generate_all()
assert len(figure_paths) == 11
display(pd.DataFrame({"generated_figure": [str(path.relative_to(ROOT)) for path in figure_paths]}))

## 5. Visual verification of all generated figures

In [ ]:
from IPython.display import Image, display

for path in figure_paths:
    print(path.name)
    display(Image(filename=str(path), width=1000))